In [5]:
import socket
import threading
import time
from collections import deque
import ipywidgets as widgets
from IPython.display import display

HOST = '0.0.0.0'
PORT = 5000
LEASE_DURATION = 5.0

state_lock = threading.Lock()
running = True
queue = deque()
current_holder = None
lease_expiry = 0.0
known_nodes = set()
server_sock = None

In [6]:
# --- GUI ---
node_widgets_cache = {}
dashboard_container = widgets.GridBox(layout=widgets.Layout(grid_template_columns="repeat(4, 150px)", gap="10px"))
header_widget = widgets.HTML("<h3>Coordinator</h3>")
info_widget = widgets.HTML("Initialisiere...")
stop_button = widgets.Button(description="Server Stoppen", button_style='danger', icon='power-off')

def get_node_ui(nid):
    lbl = widgets.HTML(value=f"<b>{nid}</b><br>IDLE", layout=widgets.Layout(height='40px', justify_content='center'))
    prog = widgets.FloatProgress(value=0.0, min=0.0, max=LEASE_DURATION, bar_style='info', layout=widgets.Layout(width='100%', height='20px'))
    box = widgets.VBox([lbl, prog], layout=widgets.Layout(border='1px solid #ccc', padding='10px', align_items='center'))
    return {'box': box, 'label': lbl, 'prog': prog}

def update_dashboard():
    while running:
        with state_lock:
            current_time = time.time()
            global current_holder, lease_expiry

            # Lease Logic
            if current_holder and current_time > lease_expiry:
                if queue:
                    next_id, next_addr = queue.popleft()
                    current_holder = next_id
                    lease_expiry = time.time() + LEASE_DURATION
                else:
                    current_holder = None

            status_text = "ONLINE" if running else "GESTOPPT"
            my_ip = socket.gethostbyname(socket.gethostname())
            info_widget.value = f"IP: {my_ip} | Port: {PORT} | Status: {status_text}"

            active_children = []
            waiting_ids = [x[0] for x in queue]

            for nid in sorted(known_nodes):
                if nid not in node_widgets_cache: node_widgets_cache[nid] = get_node_ui(nid)
                ui = node_widgets_cache[nid]
                active_children.append(ui['box'])

                if nid == current_holder:
                    rem = max(0, lease_expiry - current_time)
                    ui['label'].value = f"<b style='color:#27ae60'>{nid}</b><br>RUNNING"
                    ui['prog'].value = rem
                    ui['prog'].bar_style = 'success'
                    ui['box'].layout.border = '2px solid #27ae60'
                elif nid in waiting_ids:
                    pos = waiting_ids.index(nid) + 1
                    ui['label'].value = f"<b>{nid}</b><br>WAITING ({pos})"
                    ui['prog'].value = LEASE_DURATION
                    ui['prog'].bar_style = 'warning'
                    ui['box'].layout.border = '2px solid #f39c12'
                else:
                    ui['label'].value = f"<b>{nid}</b><br>IDLE"
                    ui['prog'].value = 0
                    ui['prog'].bar_style = ''
                    ui['box'].layout.border = '1px solid #ccc'

            if tuple(active_children) != dashboard_container.children:
                dashboard_container.children = tuple(active_children)

        time.sleep(0.1)

    info_widget.value = "<b>SYSTEM HERUNTERGEFAHREN.</b> Socket geschlossen."

In [7]:
def udp_server():
    global current_holder, lease_expiry, running, server_sock

    server_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    server_sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    try:
        server_sock.bind((HOST, PORT))
    except OSError:
        print(f"FEHLER: Port {PORT} ist noch belegt! Bitte Kernel neustarten.")
        running = False
        return

    print(f"UDP Server läuft auf {PORT}")

    while running:
        try:
            # Grants (Lease)
            with state_lock:
                 if current_holder and time.time() > lease_expiry and queue:
                     next_id, next_addr = queue.popleft()
                     current_holder = next_id
                     lease_expiry = time.time() + LEASE_DURATION
                     server_sock.sendto(b'GRANT', next_addr)

            try:
                data, addr = server_sock.recvfrom(1024)
            except socket.timeout:
                continue
            except OSError:
                break # Socket geschlossen

            msg = data.decode()

            if msg == "INTERNAL_STOP":
                print("Stop-Signal empfangen.")
                break

            parts = msg.split()
            if len(parts) < 2: continue
            cmd, node_id = parts[0], parts[1]

            with state_lock:
                known_nodes.add(node_id)
                if cmd == 'REQ':
                    if current_holder is None:
                        current_holder = node_id
                        lease_expiry = time.time() + LEASE_DURATION
                        server_sock.sendto(b'GRANT', addr)
                    elif current_holder == node_id:
                        lease_expiry = time.time() + LEASE_DURATION
                        server_sock.sendto(b'GRANT', addr)
                    else:
                        if node_id not in [x[0] for x in queue]:
                            queue.append((node_id, addr))
                elif cmd == 'HB':
                    if node_id == current_holder:
                        lease_expiry = time.time() + LEASE_DURATION
                elif cmd == 'REL':
                    if node_id == current_holder:
                        if queue:
                            next_id, next_addr = queue.popleft()
                            current_holder = next_id
                            lease_expiry = time.time() + LEASE_DURATION
                            server_sock.sendto(b'GRANT', next_addr)
                        else:
                            current_holder = None
        except Exception as e:
            if running: print(f"Server Error: {e}")

    if server_sock: server_sock.close()
    print("Server Thread beendet.")

In [8]:
def trigger_stop(b):
    global running, server_sock
    if not running: return

    print("Beende System...")
    running = False
    stop_button.disabled = True
    stop_button.description = "Stoppe..."

    try:
        temp_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        temp_sock.sendto(b"INTERNAL_STOP Node_sys", ('127.0.0.1', PORT))
        temp_sock.close()
    except Exception as e:
        print(f"Konnte Wake-Up Packet nicht senden: {e}")

    time.sleep(0.1)
    if server_sock:
        try:
            server_sock.close()
        except:
            pass

stop_button.on_click(trigger_stop)

#START
running = True
known_nodes.clear()
queue.clear()
current_holder = None
node_widgets_cache = {}

t_udp = threading.Thread(target=udp_server, daemon=True)
t_gui = threading.Thread(target=update_dashboard, daemon=True)

t_udp.start()
t_gui.start()

ui = widgets.VBox([header_widget, info_widget, stop_button, widgets.HTML("<hr>"), dashboard_container])
display(ui)

UDP Server läuft auf 5000
